# README
## Purpose
Run chronological train/validation/test BERTopic workflow on GSIB model input.
## Inputs
- `Data/GSIB_model_input.csv`
## Outputs
- Split-aware BERTopic model outputs, ranking metrics, and test-set diagnostics.
## Notes
Use this notebook for the main GSIB test-train protocol with strict temporal separation.

In [15]:
import sys, torch
print('python_executable:', sys.executable)
print('torch_version:', torch.__version__)
print('torch_cuda_runtime:', torch.version.cuda)
print('cuda_available:', torch.cuda.is_available())
print('gpu_count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu_name:', torch.cuda.get_device_name(0))

python_executable: c:\Users\gianf\AppData\Local\Programs\Python\Python313\python.exe
torch_version: 2.11.0+cu128
torch_cuda_runtime: 12.8
cuda_available: True
gpu_count: 1
gpu_name: NVIDIA GeForce RTX 3080


Imports

In [16]:
import pandas as pd
import numpy as np
import torch
from umap import UMAP
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import OnlineCountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

Load Data

In [17]:
df = pd.read_csv('Data/GSIB_model_input.csv')
print('Loaded: Data/GSIB_model_input.csv')
print('Shape:', df.shape)
display(df.head())

Loaded: Data/GSIB_model_input.csv
Shape: (24013, 5)


,stock,date,date_only,headline,headline_raw
0,NDAQ,2025-04-09 00:00:19+00:00,2025-04-09,"Dave Portnoy Says What Many Are Thinking, 'If ...","Dave Portnoy Says What Many Are Thinking, 'If ..."
1,SAN,2025-04-09 09:17:00+00:00,2025-04-09,New Strong Buy Stocks for April 9th,New Strong Buy Stocks for April 9th
2,BNPQY,2025-04-09 09:19:38+00:00,2025-04-09,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...
3,DB,2025-04-09 09:19:38+00:00,2025-04-09,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...,Treasuries ‘Fire Sale’ Sends Long-Term Yields ...
4,DB,2025-04-09 12:32:54+00:00,2025-04-09,"Bessent Sees ‘Normal Deleveraging’ in Bonds, W...","Bessent Sees ‘Normal Deleveraging’ in Bonds, W..."


In [18]:
# 1. Parse date and prepare chronological data
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date', 'headline']).sort_values('date').reset_index(drop=True)

# Deduplicate semantically identical headlines (topic modeling should not overcount copies across tickers)
before_dedup = len(df)
df = df.drop_duplicates(subset=['headline']).sort_values('date').reset_index(drop=True)
after_dedup = len(df)

# 2. Define split proportions and embargo
SPLIT_TRAIN = 0.70
SPLIT_VAL = 0.25
SPLIT_TEST = 0.05

# 3. Calculate date-based boundaries (keep same day together)
unique_dates = np.array(sorted(df['date'].dt.floor('D').unique()))
n_dates = len(unique_dates)

train_end_idx = int(SPLIT_TRAIN * n_dates)
val_end_idx = int((SPLIT_TRAIN + SPLIT_VAL) * n_dates)

# 4. Create masks with embargo
day_code, _ = pd.factorize(df['date'].dt.floor('D'), sort=True)
mask_train = day_code < train_end_idx
mask_val = (day_code >= (train_end_idx)) & (day_code < val_end_idx)
mask_test = day_code >= (val_end_idx)

# 5. Extract splits
train_df = df[mask_train].copy()
val_df = df[mask_val].copy()
test_df = df[mask_test].copy()

# Final lists for the model
train_docs, train_timestamps = train_df['headline'].tolist(), train_df['date'].tolist()
val_docs, val_timestamps = val_df['headline'].tolist(), val_df['date'].tolist()
test_docs, test_timestamps = test_df['headline'].tolist(), test_df['date'].tolist()

print(f'Deduplication: {before_dedup} -> {after_dedup} rows')
print(f"Train size: {len(train_docs)}")
print(f"Validation size: {len(val_docs)}")
print(f"Test size: {len(test_docs)}")
print('Date ranges:')
print(f"  Train: {train_df['date'].min()} -> {train_df['date'].max()}")
print(f"  Val:   {val_df['date'].min()} -> {val_df['date'].max()}")
print(f"  Test:  {test_df['date'].min()} -> {test_df['date'].max()}")

Deduplication: 24013 -> 19178 rows
Train size: 8737
Validation size: 8437
Test size: 2004
Date ranges:
  Train: 2025-04-09 00:00:19+00:00 -> 2025-12-20 21:15:53+00:00
  Val:   2025-12-21 03:12:45+00:00 -> 2026-03-21 22:20:00+00:00
  Test:  2026-03-22 00:17:26+00:00 -> 2026-04-09 21:34:38+00:00


Prepare Models

In [ ]:


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Add placeholder tokens to stopwords so they do not dominate topics
placeholder_stopwords = {'__ticker__', '__company__', 'ticker', 'company'}
model_stopwords = sorted(set(ENGLISH_STOP_WORDS).union(placeholder_stopwords))

sentence_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
train_embeddings = sentence_model.encode(
    train_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
print(f'Embeddings computed on device: {device}')

Torch CUDA available: True
GPU: NVIDIA GeForce RTX 3080


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 949.20it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 949.20it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 137/137 [00:01<00:00, 100.62it/s]

Embeddings computed on device: cuda


Train BERTopic

In [20]:
print("Training BERTopic model on training split...\n")

umap_model = UMAP(n_neighbors=15, n_components=10, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = OnlineCountVectorizer(stop_words=model_stopwords)

topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  calculate_probabilities=True,
  verbose=True
)

# Train BERTopic only on train split
topics_train, probs_train = topic_model.fit_transform(train_docs, embeddings=train_embeddings)

print("\nModel training complete!")

2026-04-09 23:56:38,845 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Training BERTopic model on training split...



2026-04-09 23:56:52,724 - BERTopic - Dimensionality - Completed ✓
2026-04-09 23:56:52,725 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-09 23:56:52,725 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-09 23:56:58,405 - BERTopic - Cluster - Completed ✓
2026-04-09 23:56:58,409 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-09 23:56:58,405 - BERTopic - Cluster - Completed ✓
2026-04-09 23:56:58,409 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-09 23:56:58,518 - BERTopic - Representation - Completed ✓
2026-04-09 23:56:58,518 - BERTopic - Representation - Completed ✓



Model training complete!


Get Topic Information

In [21]:
# Get topic information
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,2737,-1_earnings_bank_markets_group,"[earnings, bank, markets, group, investors, st...",[T. Rowe Price __TICKER__ Expected to Beat Ear...
1,0,209,0_blackstone_blackrock_shooting_power,"[blackstone, blackrock, shooting, power, billi...",[Why Blackstone __TICKER__ Shares Are Trading ...
2,1,174,1_brookfield_asset_management_gic,"[brookfield, asset, management, gic, bn, renew...",[Brookfield Asset Management Announces Record ...
3,2,131,2_intercontinental_exchange_statistics_abaxx,"[intercontinental, exchange, statistics, abaxx...","[Intercontinental Exchange, Inc. __TICKER__ Re..."
4,3,123,3_ai_cloud_google_power,"[ai, cloud, google, power, productivity, intel...","[Google May Be Key AI And Cloud Winner, __COMP..."
...,...,...,...,...,...
164,163,11,163_right_fund_pick_mutual,"[right, fund, pick, mutual, strong, bond, stee...",[Is Invesco Diversified Dividend A (LCEAX) a S...
165,164,11,164_weight_equal_invest_etf,"[weight, equal, invest, etf, 500, rspd, discre...",[Should You Invest in the Invesco S&amp;P 500 ...
166,165,10,165_swift_tokenized_boerse_stuttgart,"[swift, tokenized, boerse, stuttgart, blockcha...","[Swift, Big Banks Are Building a Blockchain Ne..."
167,166,10,166_funds_certain_management_asset,"[funds, certain, management, asset, columbia, ...",[__TICKER__ Asset Management Inc. announces ma...


Visualize Topics

In [22]:
topic_model.visualize_documents(train_docs, embeddings=train_embeddings)

In [23]:
fig = topic_model.visualize_heatmap()
fig.show()

In [24]:
hierarchical_topics = topic_model.hierarchical_topics(train_docs)
fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig.show()

print("💡 This shows the hierarchical structure of topics")

100%|██████████| 167/167 [00:00<00:00, 570.31it/s]



💡 This shows the hierarchical structure of topics


In [25]:
import plotly.express as px

topics_over_time = topic_model.topics_over_time(
    train_docs,
    train_timestamps,
    nr_bins=20
 )

tot = topics_over_time.copy()
tot = tot[tot['Topic'] != -1].copy()
tot['Timestamp'] = pd.to_datetime(tot['Timestamp']).dt.to_period('M').dt.to_timestamp()

topic_labels = topic_model.get_topic_info()[['Topic', 'Name']].copy()
topic_labels = topic_labels[topic_labels['Topic'] != -1]
tot = tot.merge(topic_labels, on='Topic', how='left')
tot['TopicLabel'] = tot['Topic'].astype(str) + ': ' + tot['Name'].fillna('')

stacked = (
    tot.groupby(['Timestamp', 'TopicLabel'], as_index=False)['Frequency']
       .sum()
       .sort_values(['Timestamp', 'Frequency'], ascending=[True, False])
)

fig = px.bar(
    stacked,
    x='Timestamp',
    y='Frequency',
    color='TopicLabel',
    title='Topic Frequency Over Time (Stacked)',
    labels={'Timestamp': 'Month', 'Frequency': 'Document Count', 'TopicLabel': 'Topic'}
)
fig.update_layout(
    barmode='stack',
    xaxis_title='Month',
    yaxis_title='Document Count',
    legend_title='Topic',
    hovermode='x unified'
 )
fig.show()

20it [00:01, 14.99it/s]



Preparing Validation

In [26]:
bertopic_topics = [
    [word for word, _ in topic_model.get_topic(t)]
    for t in topic_model.get_topics().keys() if t != -1
    if topic_model.get_topic(t) is not None
 ]

# Project validation documents into trained topic space
val_topics, val_probs = topic_model.transform(val_docs)

print("Validation transform complete (test set remains untouched).")

Batches: 100%|██████████| 264/264 [00:02<00:00, 131.13it/s]
2026-04-09 23:57:05,547 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.

2026-04-09 23:57:05,547 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-09 23:57:17,835 - BERTopic - Dimensionality - Completed ✓
2026-04-09 23:57:17,835 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-09 23:57:17,835 - BERTopic - Dimensionality - Completed ✓
2026-04-09 23:57:17,835 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-09 23:57:18,253 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-09 23:57:18,253 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-04-09 23:57:23,736 - BERTopic - Probabilities - Completed ✓
2026-04-09 23:57:23,737 - BERTopic - Cluster - Completed ✓
2026-04-09 23:57:23,736 - BERTopic - Probabilities - Completed ✓
2026-04-09 23:57:23,737 - 

Validation transform complete (test set remains untouched).


In [27]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
import nltk
import warnings
from nltk.corpus import stopwords

# Download required NLTK data
nltk.download('stopwords', quiet=True)

# Numerical stability controls
COHERENCE_EPS = 1e-8
COHERENCE_MIN_TEXTS = 20
COHERENCE_MIN_VOCAB = 30
SIMILARITY_MAX_POINTS_PER_TOPIC = 300

# Build a tokenizer aligned with BERTopic vocabulary (supports unigrams + bigrams)
stop_words = list(model_stopwords) if 'model_stopwords' in globals() else list(stopwords.words('english'))
coherence_vectorizer = CountVectorizer(
    stop_words=stop_words,
    ngram_range=(1, 2),
    token_pattern=r'(?u)\b\w\w+\b'
)
coherence_analyzer = coherence_vectorizer.build_analyzer()

def preprocess_documents(doc_list):
    tokenized = [coherence_analyzer(str(doc).lower()) for doc in doc_list]
    return [tokens for tokens in tokenized if len(tokens) >= 2]

processed_train_docs = preprocess_documents(train_docs)
processed_val_docs = preprocess_documents(val_docs)

# Dictionary from train split only
id2word = Dictionary(processed_train_docs)

def topic_diversity(topics):
    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)
    return len(unique_words) / len(all_words) if len(all_words) > 0 else 0.0

def _filter_topics_for_dictionary(topics, dictionary, min_words_per_topic=3):
    vocab = dictionary.token2id
    filtered = []
    for topic in topics:
        cleaned = []
        for term in topic:
            token = str(term).strip().lower()
            if token in vocab:
                cleaned.append(token)
        cleaned = list(dict.fromkeys(cleaned))
        if len(cleaned) >= min_words_per_topic:
            filtered.append(cleaned)
    return filtered

def safe_silhouette_score(
    embeddings,
    labels,
    outlier_label=-1,
    metric='cosine',
    min_cluster_size_for_metric=2,
    min_points_for_metric=10,
    drop_singletons=True
):
    if embeddings is None or labels is None:
        return np.nan

    labels_arr = np.asarray(labels)
    embeddings_arr = np.asarray(embeddings)

    if embeddings_arr.ndim != 2 or labels_arr.ndim != 1:
        return np.nan
    if embeddings_arr.shape[0] != labels_arr.shape[0]:
        return np.nan

    finite_mask = np.isfinite(labels_arr.astype(float, copy=False))
    valid_mask = (labels_arr != outlier_label) & finite_mask
    if valid_mask.sum() < min_points_for_metric:
        return np.nan

    labels_valid = labels_arr[valid_mask]
    emb_valid = embeddings_arr[valid_mask]

    if drop_singletons:
        unique_labels, counts = np.unique(labels_valid, return_counts=True)
        keep_labels = unique_labels[counts >= min_cluster_size_for_metric]
        if len(keep_labels) < 2:
            return np.nan
        keep_mask = np.isin(labels_valid, keep_labels)
        labels_valid = labels_valid[keep_mask]
        emb_valid = emb_valid[keep_mask]

    unique_labels, counts = np.unique(labels_valid, return_counts=True)
    if len(unique_labels) < 2:
        return np.nan
    if np.any(counts < min_cluster_size_for_metric):
        return np.nan
    if len(labels_valid) < max(min_points_for_metric, len(unique_labels) + 1):
        return np.nan

    try:
        value = float(silhouette_score(emb_valid, labels_valid, metric=metric))
        return value if np.isfinite(value) else np.nan
    except Exception:
        return np.nan

def intra_inter_topic_similarity(
    embeddings,
    labels,
    outlier_label=-1,
    max_points_per_topic=SIMILARITY_MAX_POINTS_PER_TOPIC,
    random_state=42
):
    if embeddings is None or labels is None:
        return np.nan, np.nan

    labels_arr = np.asarray(labels)
    embeddings_arr = np.asarray(embeddings)

    if embeddings_arr.ndim != 2 or labels_arr.ndim != 1:
        return np.nan, np.nan
    if embeddings_arr.shape[0] != labels_arr.shape[0]:
        return np.nan, np.nan

    finite_mask = np.isfinite(labels_arr.astype(float, copy=False))
    valid_mask = (labels_arr != outlier_label) & finite_mask
    if valid_mask.sum() < 2:
        return np.nan, np.nan

    labels_valid = labels_arr[valid_mask]
    emb_valid = embeddings_arr[valid_mask]
    rng = np.random.default_rng(random_state)

    intra_values = []
    centroids = []

    for topic_label in np.unique(labels_valid):
        topic_mask = labels_valid == topic_label
        topic_emb = emb_valid[topic_mask]
        if topic_emb.shape[0] == 0:
            continue

        if topic_emb.shape[0] > max_points_per_topic:
            sampled_idx = rng.choice(topic_emb.shape[0], size=max_points_per_topic, replace=False)
            topic_emb = topic_emb[sampled_idx]

        centroids.append(topic_emb.mean(axis=0))

        if topic_emb.shape[0] >= 2:
            sim_matrix = cosine_similarity(topic_emb)
            upper_idx = np.triu_indices(sim_matrix.shape[0], k=1)
            if len(upper_idx[0]) > 0:
                intra_values.append(float(np.nanmean(sim_matrix[upper_idx])))

    intra_topic = float(np.nanmean(intra_values)) if len(intra_values) > 0 else np.nan

    if len(centroids) < 2:
        inter_topic = np.nan
    else:
        centroids_arr = np.vstack(centroids)
        centroid_sim = cosine_similarity(centroids_arr)
        upper_idx = np.triu_indices(centroid_sim.shape[0], k=1)
        inter_topic = float(np.nanmean(centroid_sim[upper_idx])) if len(upper_idx[0]) > 0 else np.nan

    return intra_topic, inter_topic

def coherence_score(topics, texts, dictionary, coherence_type='c_v', jitter=COHERENCE_EPS):
    clean_texts = [t for t in texts if isinstance(t, list) and len(t) >= 2]
    if len(clean_texts) < COHERENCE_MIN_TEXTS or len(dictionary) < COHERENCE_MIN_VOCAB:
        return np.nan if jitter <= 0 else float(jitter)

    local_dict = Dictionary(clean_texts)
    local_dict.filter_extremes(no_below=2, no_above=0.95)
    local_dict.compactify()
    if len(local_dict) < 20:
        return np.nan if jitter <= 0 else float(jitter)

    filtered_topics = _filter_topics_for_dictionary(topics, local_dict, min_words_per_topic=3)
    if len(filtered_topics) < 2:
        return np.nan if jitter <= 0 else float(jitter)

    def _compute_value(kind):
        if kind == 'u_mass':
            corpus = [local_dict.doc2bow(doc) for doc in clean_texts]
            cm = CoherenceModel(
                topics=filtered_topics,
                corpus=corpus,
                dictionary=local_dict,
                coherence='u_mass'
            )
        else:
            cm = CoherenceModel(
                topics=filtered_topics,
                texts=clean_texts,
                dictionary=local_dict,
                coherence=kind
            )
        return float(cm.get_coherence())

    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=RuntimeWarning, module='gensim')
        with np.errstate(divide='ignore', invalid='ignore'):
            try:
                value = _compute_value(coherence_type)
                if np.isfinite(value):
                    return float(value)
            except Exception:
                pass

            # Only c_v gets u_mass fallback. For c_npmi we avoid mixed-scale fallback.
            if coherence_type == 'c_v':
                try:
                    fallback_value = _compute_value('u_mass')
                    if np.isfinite(fallback_value):
                        return float(fallback_value)
                except Exception:
                    pass

    return np.nan if jitter <= 0 else float(jitter)

cv_train = coherence_score(bertopic_topics, processed_train_docs, id2word, 'c_v')
npmi_train = coherence_score(bertopic_topics, processed_train_docs, id2word, 'c_npmi')
cv_val = coherence_score(bertopic_topics, processed_val_docs, id2word, 'c_v')
npmi_val = coherence_score(bertopic_topics, processed_val_docs, id2word, 'c_npmi')
div_train = topic_diversity(bertopic_topics)
val_outlier_ratio = float(np.mean(np.array(val_topics) == -1)) if len(val_topics) > 0 else np.nan
valid_topic_count = len(_filter_topics_for_dictionary(bertopic_topics, id2word, min_words_per_topic=3))

val_embeddings_for_metrics = sentence_model.encode(
    val_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
sil_val = safe_silhouette_score(val_embeddings_for_metrics, val_topics)
sil_val = 0.0 if not np.isfinite(sil_val) else sil_val
intra_val, inter_val = intra_inter_topic_similarity(val_embeddings_for_metrics, val_topics)

print(f"BERTopic Coherence C_v (train): {cv_train:.4f}")
print(f"BERTopic Coherence NPMI (train): {npmi_train:.4f}")
print(f"BERTopic Coherence C_v (val):   {cv_val:.4f}")
print(f"BERTopic Coherence NPMI (val):   {npmi_val:.4f}")
print(f"BERTopic Topic Diversity (train): {div_train:.4f}")
print(f"Validation outlier ratio (-1 topics): {val_outlier_ratio:.4f}")
print(f"Validation silhouette (cosine, no outliers): {sil_val:.4f}")
print(f"Validation intra-topic similarity: {intra_val:.4f}")
print(f"Validation inter-topic similarity: {inter_val:.4f}")
print(f"Valid topic count for coherence: {valid_topic_count}")

Batches: 100%|██████████| 132/132 [00:01<00:00, 88.18it/s] 



BERTopic Coherence C_v (train): 0.5290
BERTopic Coherence NPMI (train): 0.1029
BERTopic Coherence C_v (val):   0.4156
BERTopic Coherence NPMI (val):   -0.1206
BERTopic Topic Diversity (train): 0.7411
Validation outlier ratio (-1 topics): 0.4676
Validation silhouette (cosine, no outliers): 0.0710
Validation intra-topic similarity: 0.4917
Validation inter-topic similarity: 0.4120
Valid topic count for coherence: 168


Hyperparameter tuning with rolling time-series CV (train+val only, fixed seed) and bucket-wise reporting

## Tuning Strategy & Composite Scoring Explained

This cell runs a **comprehensive hyperparameter search** with a Jehnen-style multi-metric ranking setup.

### Search Design
- **Space**: 300 targeted parameter combinations (n_neighbors × n_components × min_cluster_size × min_samples × ngram_range).
- **Folds**: 3 rolling/expanding time-series CV folds on train+val (test untouched).
- **Seed**: Fixed `RANDOM_SEED=42` for deterministic model selection.

### Per-Fold Metrics Used for Ranking (Equal Weighting)
| Metric | Direction | Notes |
|--------|-----------|-------|
| **c_v coherence** | Higher is better | Semantic coherence of topic words |
| **NPMI coherence** | Higher is better | Strong coherence signal for topic quality |
| **Topic diversity** | Higher is better | Fraction of unique words across topics |
| **Intra-topic similarity** | Higher is better | Cohesion within topic clusters in embedding space |
| **Inter-topic similarity** | Lower is better | Separation between topic clusters in embedding space |

### Aggregation & Normalization
1. Average each metric across folds per hyperparameter combination.
2. Min-max normalize each metric to [0,1].
3. Invert inter-topic similarity during normalization so lower raw inter-topic similarity gets higher normalized score.

### Composite Scoring Formula (No Manual Weighting)
All five ranking metrics are weighted equally:

$$
\text{score} = \frac{1}{5}\left(
\text{c\_v}_{\text{norm}} + \text{npmi}_{\text{norm}} + \text{diversity}_{\text{norm}} + \text{intra}_{\text{norm}} + \text{inter\_separation}_{\text{norm}}
\right)
$$

This is equivalent to **no preferential weighting** between ranking metrics.

### Model Selection
The highest composite score row is selected as `best_params`. Fold-level best rows are also reported to inspect temporal stability.

### Next: Multi-Seed Final Evaluation
After selecting best params, the final model is re-fit and evaluated across 9 random seeds on the untouched test set.

In [28]:
from itertools import product
import time

# ================================================================================
# HYPERPARAMETER TUNING: GRID SEARCH + ROLLING CV + JEHNEN-STYLE EQUAL-WEIGHT RANKING
# ================================================================================
# Ranking metrics (equal weight):
#   - c_v coherence
#   - c_npmi coherence
#   - topic diversity
#   - intra-topic similarity (higher better)
#   - inter-topic similarity (lower better; inverted during normalization)
# ================================================================================

# -------------------- Time-series CV controls (fixed seed for model selection) --------------------
N_FOLDS = 3  # rolling/expanding buckets before test
RANDOM_SEED = 42

# Build tuning pool from train + validation only (test remains untouched)
tune_df = pd.concat([train_df, val_df], axis=0).sort_values('date').reset_index(drop=True)
tune_docs = tune_df['headline'].tolist()
tune_timestamps = tune_df['date'].tolist()

# Precompute embeddings for the full tuning pool once
tune_embeddings = sentence_model.encode(
    tune_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)

def sanitize_metric(value, fallback=np.nan):
    try:
        v = float(value)
    except Exception:
        return fallback
    return v if np.isfinite(v) else fallback

def make_expanding_folds(df_dates, n_folds=3, embargo_n=1):
    unique_days = np.array(sorted(pd.to_datetime(df_dates).dt.floor('D').unique()))
    n_days = len(unique_days)
    edges = np.linspace(0, n_days, n_folds + 2, dtype=int)
    day_code, _ = pd.factorize(pd.to_datetime(df_dates).dt.floor('D'), sort=True)

    folds = []
    for fold_idx in range(n_folds):
        train_end = edges[fold_idx + 1]
        val_start = train_end + embargo_n
        val_end = edges[fold_idx + 2]

        if train_end <= 0 or val_start >= val_end:
            continue

        train_mask = day_code < train_end
        val_mask = (day_code >= val_start) & (day_code < val_end)

        train_idx = np.where(train_mask)[0]
        val_idx = np.where(val_mask)[0]

        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        folds.append({
            'fold': fold_idx + 1,
            'train_idx': train_idx,
            'val_idx': val_idx,
            'train_start': unique_days[0],
            'train_end': unique_days[train_end - 1],
            'val_start': unique_days[val_start],
            'val_end': unique_days[val_end - 1]
        })

    return folds

folds = make_expanding_folds(tune_df['date'], n_folds=N_FOLDS)
print(f'Generated {len(folds)} rolling folds (train+val pool only).')
for fold in folds:
    print(
        f"Fold {fold['fold']}: train {fold['train_start']} -> {fold['train_end']} | "
        f"val {fold['val_start']} -> {fold['val_end']}"
    )

param_grid = {
    'n_neighbors': [10, 30, 50, 70, 90],
    'n_components': [4, 6, 8],
    'min_cluster_size': [8, 12, 16, 24, 32],
    'min_samples': [1, 2, 4, 6],
    'ngram_range': [(1, 2)]
}

search_space = list(product(
    param_grid['n_neighbors'],
    param_grid['n_components'],
    param_grid['min_cluster_size'],
    param_grid['min_samples'],
    param_grid['ngram_range']
))
expected_combinations = (
    len(param_grid['n_neighbors'])
    * len(param_grid['n_components'])
    * len(param_grid['min_cluster_size'])
    * len(param_grid['min_samples'])
    * len(param_grid['ngram_range'])
)

print(f'Configured combinations: {expected_combinations} (target ~300)')
print(f'Running {len(search_space)} parameter combinations x {len(folds)} folds (fixed seed={RANDOM_SEED})')

cv_rows = []

for trial, (n_neighbors, n_components, min_cluster_size, min_samples, ngram_range) in enumerate(search_space, start=1):
    print(
        f"[Trial {trial}/{len(search_space)}] n_neighbors={n_neighbors}, n_components={n_components}, "
        f"min_cluster_size={min_cluster_size}, min_samples={min_samples}, ngram_range={ngram_range}"
    )

    for fold in folds:
        train_idx = fold['train_idx']
        val_idx = fold['val_idx']

        fold_train_docs = [tune_docs[i] for i in train_idx]
        fold_val_docs = [tune_docs[i] for i in val_idx]
        fold_train_embeddings = tune_embeddings[train_idx]
        fold_val_embeddings = tune_embeddings[val_idx]

        processed_train_fold = preprocess_documents(fold_train_docs)
        processed_val_fold = preprocess_documents(fold_val_docs)
        id2word_fold = Dictionary(processed_train_fold)

        start = time.perf_counter()

        umap_model_i = UMAP(
            n_neighbors=n_neighbors,
            n_components=n_components,
            metric='cosine',
            random_state=RANDOM_SEED
        )
        hdbscan_model_i = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric='euclidean',
            cluster_selection_method='eom',
            prediction_data=True,
            core_dist_n_jobs=-1
        )
        vectorizer_model_i = OnlineCountVectorizer(stop_words=model_stopwords, ngram_range=ngram_range)

        topic_model_i = BERTopic(
            embedding_model=sentence_model,
            umap_model=umap_model_i,
            hdbscan_model=hdbscan_model_i,
            vectorizer_model=vectorizer_model_i,
            calculate_probabilities=False,
            verbose=False
        )

        _topics_train_i, _ = topic_model_i.fit_transform(fold_train_docs, embeddings=fold_train_embeddings)
        val_topics_i, _ = topic_model_i.transform(fold_val_docs, embeddings=fold_val_embeddings)

        topic_words_i = [
            [word for word, _ in topic_model_i.get_topic(t)]
            for t in topic_model_i.get_topics().keys()
            if t != -1 and topic_model_i.get_topic(t) is not None
        ]

        has_enough_for_coherence = (
            len(processed_val_fold) >= 20 and
            len(id2word_fold) >= 30 and
            len(topic_words_i) >= 2
        )

        cv_val_i = sanitize_metric(
            coherence_score(topic_words_i, processed_val_fold, id2word_fold, 'c_v', jitter=COHERENCE_EPS),
            fallback=COHERENCE_EPS
        ) if has_enough_for_coherence else COHERENCE_EPS

        npmi_val_i = sanitize_metric(
            coherence_score(topic_words_i, processed_val_fold, id2word_fold, 'c_npmi', jitter=COHERENCE_EPS),
            fallback=COHERENCE_EPS
        ) if has_enough_for_coherence else COHERENCE_EPS

        div_i = sanitize_metric(topic_diversity(topic_words_i), fallback=0.0)
        intra_i, inter_i = intra_inter_topic_similarity(fold_val_embeddings, val_topics_i)
        intra_i = sanitize_metric(intra_i, fallback=0.0)
        inter_i = sanitize_metric(inter_i, fallback=1.0)

        n_topics_i = int(sum(1 for t in topic_model_i.get_topics().keys() if int(t) != -1))
        elapsed = time.perf_counter() - start

        cv_rows.append({
            'trial': trial,
            'fold': fold['fold'],
            'random_seed': RANDOM_SEED,
            'n_neighbors': n_neighbors,
            'n_components': n_components,
            'min_cluster_size': min_cluster_size,
            'min_samples': min_samples,
            'ngram_range': str(ngram_range),
            'n_topics': n_topics_i,
            'cv_val': cv_val_i,
            'npmi_val': npmi_val_i,
            'topic_diversity': div_i,
            'intra_topic_similarity': intra_i,
            'inter_topic_similarity': inter_i,
            'fit_seconds': elapsed
        })

cv_results = pd.DataFrame(cv_rows)

agg_cols = [
    'cv_val', 'npmi_val', 'topic_diversity',
    'intra_topic_similarity', 'inter_topic_similarity',
    'n_topics', 'fit_seconds'
]
tuning_results = cv_results.groupby(
    ['n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range'],
    as_index=False
)[agg_cols].mean()

def minmax_norm(series, higher_is_better=True):
    s = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan)
    if s.dropna().empty:
        return pd.Series(0.0, index=s.index)
    filled = s.fillna(s.min() if higher_is_better else s.max())
    min_v, max_v = float(filled.min()), float(filled.max())
    if max_v == min_v:
        return pd.Series(0.0, index=filled.index)
    norm = (filled - min_v) / (max_v - min_v)
    return norm

tuning_results['cv_val_norm'] = minmax_norm(tuning_results['cv_val'], higher_is_better=True)
tuning_results['npmi_val_norm'] = minmax_norm(tuning_results['npmi_val'], higher_is_better=True)
tuning_results['topic_diversity_norm'] = minmax_norm(tuning_results['topic_diversity'], higher_is_better=True)
tuning_results['intra_topic_similarity_norm'] = minmax_norm(tuning_results['intra_topic_similarity'], higher_is_better=True)
tuning_results['inter_topic_separation_norm'] = minmax_norm(tuning_results['inter_topic_similarity'], higher_is_better=False)

# Equal weighting across Jehnen-style ranking metrics (no preferential weighting)
tuning_results['composite_score'] = (
    tuning_results['cv_val_norm'] +
    tuning_results['npmi_val_norm'] +
    tuning_results['topic_diversity_norm'] +
    tuning_results['intra_topic_similarity_norm'] +
    tuning_results['inter_topic_separation_norm']
) / 5.0

tuning_results = tuning_results.sort_values('composite_score', ascending=False).reset_index(drop=True)
best_params = tuning_results.iloc[0].to_dict()

# Bucket-wise (fold-wise) best params to inspect drift over time
fold_agg = cv_results.groupby(
    ['fold', 'n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range'],
    as_index=False
)[['cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity', 'n_topics']].mean()

fold_agg['cv_val_norm'] = fold_agg.groupby('fold')['cv_val'].transform(lambda x: minmax_norm(x, True))
fold_agg['npmi_val_norm'] = fold_agg.groupby('fold')['npmi_val'].transform(lambda x: minmax_norm(x, True))
fold_agg['topic_diversity_norm'] = fold_agg.groupby('fold')['topic_diversity'].transform(lambda x: minmax_norm(x, True))
fold_agg['intra_topic_similarity_norm'] = fold_agg.groupby('fold')['intra_topic_similarity'].transform(lambda x: minmax_norm(x, True))
fold_agg['inter_topic_separation_norm'] = fold_agg.groupby('fold')['inter_topic_similarity'].transform(lambda x: minmax_norm(x, False))

fold_agg['fold_score'] = (
    fold_agg['cv_val_norm'] +
    fold_agg['npmi_val_norm'] +
    fold_agg['topic_diversity_norm'] +
    fold_agg['intra_topic_similarity_norm'] +
    fold_agg['inter_topic_separation_norm']
) / 5.0

best_per_fold = (
    fold_agg.sort_values(['fold', 'fold_score'], ascending=[True, False])
            .groupby('fold', as_index=False)
            .head(1)
            .reset_index(drop=True)
)

print('\nTop configurations (overall):')
display(tuning_results.head(10))

print('\nBest params by time bucket (fold):')
display(best_per_fold[['fold', 'n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range', 'cv_val', 'npmi_val', 'topic_diversity', 'intra_topic_similarity', 'inter_topic_similarity', 'fold_score']])

print('\nAverage fit time per model run (seconds):', round(float(cv_results['fit_seconds'].mean()), 2))
print('Best overall configuration selected from rolling CV (fixed-seed model selection):')
display(pd.DataFrame([best_params]))

Batches: 100%|██████████| 269/269 [00:02<00:00, 107.59it/s]



Generated 3 rolling folds (train+val pool only).
Fold 1: train 2025-04-09 00:00:00+00:00 -> 2025-07-03 00:00:00+00:00 | val 2025-07-05 00:00:00+00:00 -> 2025-09-28 00:00:00+00:00
Fold 2: train 2025-04-09 00:00:00+00:00 -> 2025-09-28 00:00:00+00:00 | val 2025-09-30 00:00:00+00:00 -> 2025-12-24 00:00:00+00:00
Fold 3: train 2025-04-09 00:00:00+00:00 -> 2025-12-24 00:00:00+00:00 | val 2025-12-26 00:00:00+00:00 -> 2026-03-21 00:00:00+00:00
Configured combinations: 300 (target ~300)
Running 300 parameter combinations x 3 folds (fixed seed=42)
[Trial 1/300] n_neighbors=10, n_components=4, min_cluster_size=8, min_samples=1, ngram_range=(1, 2)
[Trial 2/300] n_neighbors=10, n_components=4, min_cluster_size=8, min_samples=2, ngram_range=(1, 2)
[Trial 2/300] n_neighbors=10, n_components=4, min_cluster_size=8, min_samples=2, ngram_range=(1, 2)
[Trial 3/300] n_neighbors=10, n_components=4, min_cluster_size=8, min_samples=4, ngram_range=(1, 2)
[Trial 3/300] n_neighbors=10, n_components=4, min_cluster

,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,npmi_val,topic_diversity,intra_topic_similarity,inter_topic_similarity,n_topics,fit_seconds,cv_val_norm,npmi_val_norm,topic_diversity_norm,intra_topic_similarity_norm,inter_topic_separation_norm,composite_score
0,90,8,16,6,"(1, 2)",0.593051,0.108674,0.910830,0.456183,0.480510,57.666667,48.269913,0.990571,0.994026,0.674835,0.477390,0.626713,0.752707
1,90,4,16,6,"(1, 2)",0.592433,0.106501,0.912004,0.461291,0.464665,59.666667,48.515375,0.987306,0.980823,0.687188,0.503224,0.567793,0.745267
2,90,4,16,4,"(1, 2)",0.590975,0.102159,0.905066,0.464000,0.471926,62.000000,48.214052,0.979601,0.954438,0.614163,0.516924,0.594795,0.731984
3,70,4,16,6,"(1, 2)",0.594835,0.109657,0.901620,0.446922,0.486612,59.333333,46.654636,1.000000,1.000000,0.577894,0.430555,0.649404,0.731570
4,90,6,24,6,"(1, 2)",0.563490,0.076159,0.939444,0.449512,0.472857,43.333333,48.502720,0.834357,0.796464,0.976018,0.443654,0.598257,0.729750
5,90,4,32,4,"(1, 2)",0.562458,0.082011,0.920656,0.426467,0.513682,36.333333,48.024193,0.828907,0.832021,0.778262,0.327107,0.750062,0.703272
6,90,6,32,6,"(1, 2)",0.541607,0.060966,0.941723,0.434238,0.482824,32.333333,48.433527,0.718717,0.704152,1.000000,0.366408,0.635316,0.684919
7,90,4,32,6,"(1, 2)",0.550879,0.075523,0.925935,0.426827,0.498659,35.000000,47.781132,0.767719,0.792600,0.833822,0.328926,0.694200,0.683453
8,90,4,24,6,"(1, 2)",0.556201,0.070337,0.927112,0.433098,0.486764,45.333333,47.784268,0.795840,0.761093,0.846214,0.360640,0.649968,0.682751
9,70,4,32,6,"(1, 2)",0.559733,0.071769,0.924225,0.410804,0.516621,33.666667,46.480659,0.814508,0.769793,0.815825,0.247889,0.760988,0.681800



Best params by time bucket (fold):


,fold,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,npmi_val,topic_diversity,intra_topic_similarity,inter_topic_similarity,fold_score
0,1,90,4,16,6,"(1, 2)",0.740645,0.277620,0.933333,0.398837,0.661221,0.761081
1,2,90,4,32,4,"(1, 2)",0.573908,0.077299,0.893548,0.478309,0.417280,0.724782
2,3,90,6,16,6,"(1, 2)",0.536080,0.045840,0.879091,0.534814,0.365084,0.671579



Average fit time per model run (seconds): 44.71
Best overall configuration selected from rolling CV (fixed-seed model selection):


,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,npmi_val,topic_diversity,intra_topic_similarity,inter_topic_similarity,n_topics,fit_seconds,cv_val_norm,npmi_val_norm,topic_diversity_norm,intra_topic_similarity_norm,inter_topic_separation_norm,composite_score
0,90,8,16,6,"(1, 2)",0.593051,0.108674,0.91083,0.456183,0.48051,57.666667,48.269913,0.990571,0.994026,0.674835,0.47739,0.626713,0.752707


Final model and test evaluation with multi-seed variability analysis (single-touch per seed on test split)

In [29]:
from ast import literal_eval

# Refit best hyperparameter configuration on train+val, then evaluate variability across random seeds on test
EVAL_SEEDS = [42, 7, 123, 2024, 99, 11, 77, 314, 2718]

train_val_docs = train_docs + val_docs
train_val_embeddings = sentence_model.encode(
    train_val_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
test_embeddings = sentence_model.encode(
    test_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)

processed_train_val_docs = preprocess_documents(train_val_docs)
processed_test_docs = preprocess_documents(test_docs)
id2word_train_val = Dictionary(processed_train_val_docs)

best_ngram = literal_eval(best_params['ngram_range'])
seed_rows = []

for seed in EVAL_SEEDS:
    print(f'Running final fit/eval for seed={seed}...')

    best_umap = UMAP(
        n_neighbors=int(best_params['n_neighbors']),
        n_components=int(best_params['n_components']),
        metric='cosine',
        random_state=seed
    )
    best_hdbscan = HDBSCAN(
        min_cluster_size=int(best_params['min_cluster_size']),
        min_samples=int(best_params['min_samples']),
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )
    best_vectorizer = OnlineCountVectorizer(stop_words=model_stopwords, ngram_range=best_ngram)

    best_topic_model = BERTopic(
        embedding_model=sentence_model,
        umap_model=best_umap,
        hdbscan_model=best_hdbscan,
        vectorizer_model=best_vectorizer,
        calculate_probabilities=True,
        verbose=False
    )

    _topics_train_val, _ = best_topic_model.fit_transform(train_val_docs, embeddings=train_val_embeddings)
    test_topics, _ = best_topic_model.transform(test_docs, embeddings=test_embeddings)

    best_topic_words = [
        [word for word, _ in best_topic_model.get_topic(t)]
        for t in best_topic_model.get_topics().keys()
        if t != -1 and best_topic_model.get_topic(t) is not None
    ]

    cv_test = sanitize_metric(
        coherence_score(best_topic_words, processed_test_docs, id2word_train_val, 'c_v', jitter=COHERENCE_EPS),
        fallback=COHERENCE_EPS
    )
    npmi_test = sanitize_metric(
        coherence_score(best_topic_words, processed_test_docs, id2word_train_val, 'c_npmi', jitter=COHERENCE_EPS),
        fallback=COHERENCE_EPS
    )
    div_test = sanitize_metric(topic_diversity(best_topic_words), fallback=0.0)
    intra_test, inter_test = intra_inter_topic_similarity(test_embeddings, test_topics)
    intra_test = sanitize_metric(intra_test, fallback=0.0)
    inter_test = sanitize_metric(inter_test, fallback=1.0)

    # Final-only diagnostics (not used in tuning)
    sil_test = sanitize_metric(
        safe_silhouette_score(test_embeddings, test_topics, outlier_label=-1, metric='cosine'),
        fallback=0.0
    )
    test_outlier_ratio = float(np.mean(np.array(test_topics) == -1)) if len(test_topics) > 0 else np.nan
    test_topic_count = int(sum(1 for t in best_topic_model.get_topics().keys() if int(t) != -1))

    seed_rows.append({
        'seed': seed,
        'cv_test': cv_test,
        'npmi_test': npmi_test,
        'topic_diversity_test': div_test,
        'intra_topic_similarity_test': intra_test,
        'inter_topic_similarity_test': inter_test,
        'test_silhouette': sil_test,
        'test_outlier_ratio': test_outlier_ratio,
        'test_topic_count': test_topic_count
    })

seed_results = pd.DataFrame(seed_rows)

print('\nPer-seed test metrics:')
display(seed_results)

summary_stats = pd.DataFrame([{
    'cv_test_mean': seed_results['cv_test'].mean(skipna=True),
    'cv_test_std': seed_results['cv_test'].std(skipna=True),
    'npmi_test_mean': seed_results['npmi_test'].mean(skipna=True),
    'npmi_test_std': seed_results['npmi_test'].std(skipna=True),
    'topic_diversity_mean': seed_results['topic_diversity_test'].mean(skipna=True),
    'topic_diversity_std': seed_results['topic_diversity_test'].std(skipna=True),
    'intra_topic_similarity_mean': seed_results['intra_topic_similarity_test'].mean(skipna=True),
    'intra_topic_similarity_std': seed_results['intra_topic_similarity_test'].std(skipna=True),
    'inter_topic_similarity_mean': seed_results['inter_topic_similarity_test'].mean(skipna=True),
    'inter_topic_similarity_std': seed_results['inter_topic_similarity_test'].std(skipna=True),
    'test_silhouette_mean': seed_results['test_silhouette'].mean(skipna=True),
    'test_silhouette_std': seed_results['test_silhouette'].std(skipna=True),
    'test_outlier_ratio_mean': seed_results['test_outlier_ratio'].mean(skipna=True),
    'test_outlier_ratio_std': seed_results['test_outlier_ratio'].std(skipna=True),
    'test_topic_count_mean': seed_results['test_topic_count'].mean(skipna=True),
    'test_topic_count_std': seed_results['test_topic_count'].std(skipna=True)
}])

print('\nFinal test metrics variability across random seeds:')
display(summary_stats)

def format_mean_std(mean_value, std_value, digits=4):
    if not np.isfinite(mean_value):
        return 'NA'
    if not np.isfinite(std_value):
        return f"{mean_value:.{digits}f} ± NA"
    return f"{mean_value:.{digits}f} ± {std_value:.{digits}f}"

thesis_report = pd.DataFrame([{
    'Model': 'BERTopic (best CV config)',
    'Seeds (n)': len(EVAL_SEEDS),
    'Coherence C_v (test)': format_mean_std(summary_stats.at[0, 'cv_test_mean'], summary_stats.at[0, 'cv_test_std'], digits=4),
    'Coherence NPMI (test)': format_mean_std(summary_stats.at[0, 'npmi_test_mean'], summary_stats.at[0, 'npmi_test_std'], digits=4),
    'Topic Diversity (test)': format_mean_std(summary_stats.at[0, 'topic_diversity_mean'], summary_stats.at[0, 'topic_diversity_std'], digits=4),
    'Intra-topic Similarity (test)': format_mean_std(summary_stats.at[0, 'intra_topic_similarity_mean'], summary_stats.at[0, 'intra_topic_similarity_std'], digits=4),
    'Inter-topic Similarity (test)': format_mean_std(summary_stats.at[0, 'inter_topic_similarity_mean'], summary_stats.at[0, 'inter_topic_similarity_std'], digits=4),
    'Silhouette (test, cosine)': format_mean_std(summary_stats.at[0, 'test_silhouette_mean'], summary_stats.at[0, 'test_silhouette_std'], digits=4),
    'Outlier Ratio (test)': format_mean_std(summary_stats.at[0, 'test_outlier_ratio_mean'], summary_stats.at[0, 'test_outlier_ratio_std'], digits=4),
    'Topic Count (test)': format_mean_std(summary_stats.at[0, 'test_topic_count_mean'], summary_stats.at[0, 'test_topic_count_std'], digits=2)
}])

print('\nThesis-ready reporting table (mean ± std across seeds):')
display(thesis_report)

Batches: 100%|██████████| 32/32 [00:00<00:00, 126.99it/s]



Running final fit/eval for seed=42...
Running final fit/eval for seed=7...
Running final fit/eval for seed=7...
Running final fit/eval for seed=123...
Running final fit/eval for seed=123...
Running final fit/eval for seed=2024...
Running final fit/eval for seed=2024...
Running final fit/eval for seed=99...
Running final fit/eval for seed=99...
Running final fit/eval for seed=11...
Running final fit/eval for seed=11...
Running final fit/eval for seed=77...
Running final fit/eval for seed=77...
Running final fit/eval for seed=314...
Running final fit/eval for seed=314...
Running final fit/eval for seed=2718...
Running final fit/eval for seed=2718...

Per-seed test metrics:

Per-seed test metrics:


,seed,cv_test,npmi_test,topic_diversity_test,intra_topic_similarity_test,inter_topic_similarity_test,test_silhouette,test_outlier_ratio,test_topic_count
0,42,0.518836,-0.028751,0.851572,0.506777,0.312928,0.131024,0.572854,159
1,7,0.530901,-0.010081,0.843125,0.521197,0.311975,0.141912,0.597804,160
2,123,0.531793,-0.009008,0.850299,0.529705,0.327297,0.128711,0.592814,167
3,2024,0.514188,-0.030150,0.847205,0.510962,0.325918,0.135421,0.599301,161
4,99,0.512815,-0.025607,0.853416,0.518913,0.315939,0.118848,0.561876,161
5,11,0.506962,-0.034190,0.848503,0.506039,0.311379,0.116180,0.563872,167
6,77,0.501651,-0.042325,0.844099,0.517034,0.315750,0.119859,0.571856,161
7,314,0.515200,-0.030000,0.854438,0.521184,0.319854,0.122039,0.585329,169
8,2718,0.528514,-0.010301,0.851250,0.517836,0.323143,0.117032,0.576347,160



Final test metrics variability across random seeds:


,cv_test_mean,cv_test_std,npmi_test_mean,npmi_test_std,topic_diversity_mean,topic_diversity_std,intra_topic_similarity_mean,intra_topic_similarity_std,inter_topic_similarity_mean,inter_topic_similarity_std,test_silhouette_mean,test_silhouette_std,test_outlier_ratio_mean,test_outlier_ratio_std,test_topic_count_mean,test_topic_count_std
0,0.517873,0.010643,-0.02449,0.011951,0.849323,0.003929,0.516628,0.007589,0.318243,0.006059,0.125669,0.009048,0.580228,0.014146,162.777778,3.767552



Thesis-ready reporting table (mean ± std across seeds):


,Model,Seeds (n),Coherence C_v (test),Coherence NPMI (test),Topic Diversity (test),Intra-topic Similarity (test),Inter-topic Similarity (test),"Silhouette (test, cosine)",Outlier Ratio (test),Topic Count (test)
0,BERTopic (best CV config),9,0.5179 ± 0.0106,-0.0245 ± 0.0120,0.8493 ± 0.0039,0.5166 ± 0.0076,0.3182 ± 0.0061,0.1257 ± 0.0090,0.5802 ± 0.0141,162.78 ± 3.77


In [30]:
import plotly.express as px

# Diagnostics use the currently available final test assignment (`test_topics`) and embeddings (`test_embeddings`).
if 'test_topics' not in globals() or 'test_embeddings' not in globals():
    raise RuntimeError('Please run Cell 24 first so `test_topics` and `test_embeddings` are available.')

labels = np.asarray(test_topics)
emb = np.asarray(test_embeddings)

if labels.shape[0] != emb.shape[0]:
    raise ValueError(f'Mismatch: labels={labels.shape[0]} vs embeddings={emb.shape[0]}')

is_outlier = labels == -1
outlier_ratio_now = float(np.mean(is_outlier)) if len(labels) > 0 else np.nan

valid_labels = labels[~is_outlier]
if len(valid_labels) > 0:
    cluster_size_series = pd.Series(valid_labels).value_counts().sort_values(ascending=False)
else:
    cluster_size_series = pd.Series(dtype='int64')

n_valid_clusters_now = int(cluster_size_series.shape[0])
singleton_topics_now = set(cluster_size_series[cluster_size_series == 1].index.tolist())
n_singletons_now = int(len(singleton_topics_now))

print('Final-clustering diagnostics (current test assignment):')
print(f'  Documents: {len(labels)}')
print(f'  Outlier ratio (-1): {outlier_ratio_now:.4f}')
print(f'  Valid clusters (excluding -1): {n_valid_clusters_now}')
print(f'  Singleton clusters: {n_singletons_now}')

if n_valid_clusters_now == 0:
    print('⚠️ No valid clusters found (all points are outliers).')
elif n_valid_clusters_now == 1:
    print('⚠️ Only one valid cluster found. Silhouette is undefined by design.')
if n_singletons_now > 0:
    print('⚠️ Singleton clusters present. Silhouette excludes them in our robust function.')

# 1) Cluster size distribution (excluding outliers)
if n_valid_clusters_now > 0:
    cluster_size_df = (
        cluster_size_series
        .rename_axis('Topic')
        .reset_index(name='Count')
        .sort_values('Count', ascending=False)
    )
    cluster_size_df['Topic'] = cluster_size_df['Topic'].astype(int)

    fig_sizes = px.bar(
        cluster_size_df,
        x='Topic',
        y='Count',
        title='Final Test Cluster Sizes (excluding outliers)',
        labels={'Topic': 'Topic ID', 'Count': 'Number of Documents'}
    )
    fig_sizes.show()
else:
    print('Cluster-size plot skipped (no valid clusters).')

# 2) 2D embedding view colored by diagnostic point type
# Recompute a 2D projection from test embeddings for visual inspection.
umap_viz = UMAP(n_neighbors=15, n_components=2, metric='cosine', random_state=42)
emb_2d = umap_viz.fit_transform(emb)

point_type = np.where(
    labels == -1,
    'Outlier (-1)',
    np.where(np.isin(labels, list(singleton_topics_now)), 'Singleton cluster', 'Valid cluster')
)

viz_df = pd.DataFrame({
    'x': emb_2d[:, 0],
    'y': emb_2d[:, 1],
    'topic': labels.astype(int),
    'point_type': point_type
})

fig_diag = px.scatter(
    viz_df,
    x='x',
    y='y',
    color='point_type',
    hover_data=['topic'],
    title='Final Test Embedding Diagnostics (Outliers / Singleton / Valid)'
)
fig_diag.update_traces(marker=dict(size=6, opacity=0.7))
fig_diag.show()

# 3) Seed-level context from the multi-seed run
if 'seed_results' in globals() and isinstance(seed_results, pd.DataFrame) and not seed_results.empty:
    seed_plot_df = seed_results.copy()

    fig_seed_outliers = px.bar(
        seed_plot_df,
        x='seed',
        y='test_outlier_ratio',
        title='Outlier Ratio by Seed',
        labels={'seed': 'Seed', 'test_outlier_ratio': 'Outlier Ratio'}
    )
    fig_seed_outliers.show()

    fig_seed_topics = px.bar(
        seed_plot_df,
        x='seed',
        y='test_topic_count',
        title='Valid Topic Count by Seed',
        labels={'seed': 'Seed', 'test_topic_count': 'Topic Count (excluding -1)'}
    )
    fig_seed_topics.show()
else:
    print('Seed-level plots skipped (run Cell 24 first to populate `seed_results`).')

Final-clustering diagnostics (current test assignment):
  Documents: 2004
  Outlier ratio (-1): 0.5763
  Valid clusters (excluding -1): 107
  Singleton clusters: 22
⚠️ Singleton clusters present. Silhouette excludes them in our robust function.


Cluster diagnostics and visual sanity checks (outliers, valid clusters, singleton clusters)